In [ ]:
import pandas as pd

In [ ]:
listings = pd.read_csv("/content/drive/MyDrive/Data Engineering/listings.csv")
reviews = pd.read_csv("/content/drive/MyDrive/Data Engineering/reviews.csv")
calendar = pd.read_csv("/content/drive/MyDrive/Data Engineering/calendar.csv")
neighbourhoods = pd.read_csv("/content/drive/MyDrive/Data Engineering/neighbourhoods.csv")

In [ ]:
for name, df in {
    "Listings": listings,
    "Reviews": reviews,
    "Calendar": calendar,
    "Neighbourhoods": neighbourhoods
}.items():
    print(name, df.shape)

Listings (453, 85)
Reviews (27892, 6)
Calendar (165345, 7)
Neighbourhoods (16, 2)


In [ ]:
datasets = {
    "Listings": listings,
    "Reviews": reviews,
    "Calendar": calendar,
    "Neighbourhoods": neighbourhoods
}

for name, df in datasets.items():

    cardinality = pd.DataFrame({
        "Column": df.columns,
        "Unique Values":
        [df[col].nunique() for col in df.columns]
    })

    display(cardinality)

,Column,Unique Values
0,id,453
1,listing_url,453
2,scrape_id,1
3,last_scraped,1
4,source,2
...,...,...
80,calculated_host_listings_count,15
81,calculated_host_listings_count_entire_homes,14
82,calculated_host_listings_count_private_rooms,8
83,calculated_host_listings_count_shared_rooms,1


,Column,Unique Values
0,listing_id,395
1,id,27892
2,date,3502
3,reviewer_id,23062
4,reviewer_name,7077
5,comments,27032


,Column,Unique Values
0,listing_id,453
1,date,365
2,available,2
3,price,0
4,adjusted_price,0
5,minimum_nights,24
6,maximum_nights,194


,Column,Unique Values
0,neighbourhood_group,0
1,neighbourhood,15


In [ ]:
for name, df in datasets.items():

    cardinality = pd.DataFrame({
        "Column": df.columns,
        "Unique Values":
        [df[col].nunique() for col in df.columns]
    })

    display(cardinality)

,Column,Unique Values
0,id,453
1,listing_url,453
2,scrape_id,1
3,last_scraped,1
4,source,2
...,...,...
80,calculated_host_listings_count,15
81,calculated_host_listings_count_entire_homes,14
82,calculated_host_listings_count_private_rooms,8
83,calculated_host_listings_count_shared_rooms,1


,Column,Unique Values
0,listing_id,395
1,id,27892
2,date,3502
3,reviewer_id,23062
4,reviewer_name,7077
5,comments,27032


,Column,Unique Values
0,listing_id,453
1,date,365
2,available,2
3,price,0
4,adjusted_price,0
5,minimum_nights,24
6,maximum_nights,194


,Column,Unique Values
0,neighbourhood_group,0
1,neighbourhood,15


In [ ]:
for name, df in datasets.items():

    missing = (
        df.isnull()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
    )

    display(missing)

,0
neighborhood_overview,100.0
host_since,100.0
host_acceptance_rate,100.0
host_response_rate,100.0
host_response_time,100.0
...,...
availability_30,0.0
calculated_host_listings_count,0.0
calculated_host_listings_count_entire_homes,0.0
calculated_host_listings_count_private_rooms,0.0


,0
comments,0.028682
listing_id,0.000000
id,0.000000
date,0.000000
reviewer_id,0.000000
reviewer_name,0.000000


,0
price,100.0
adjusted_price,100.0
listing_id,0.0
available,0.0
date,0.0
minimum_nights,0.0
maximum_nights,0.0


,0
neighbourhood_group,100.00
neighbourhood,6.25


In [ ]:
for name, df in datasets.items():

    print(df.dtypes.value_counts())

float64    31
int64      30
object     24
Name: count, dtype: int64
int64     3
object    3
Name: count, dtype: int64
int64      3
object     2
float64    2
Name: count, dtype: int64
float64    1
object     1
Name: count, dtype: int64


In [ ]:
for name, df in datasets.items():

    print(
        name,
        df.duplicated().sum()
    )

Listings 0
Reviews 0
Calendar 0
Neighbourhoods 0


In [ ]:
!pip install rapidfuzz
from rapidfuzz import fuzz

names = listings["name"].dropna().tolist()

for i in range(10):
    for j in range(i+1,10):

        score = fuzz.ratio(
            names[i],
            names[j]
        )

        if score > 95:
            print(names[i], names[j])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.7 MB/s eta 0:00:00


In [ ]:
cols = [
    "accommodates",
    "bedrooms",
    "number_of_reviews"
]

for col in cols:

    Q1 = listings[col].quantile(.25)
    Q3 = listings[col].quantile(.75)

    IQR = Q3-Q1

    outliers = listings[
        (listings[col] < Q1 - 1.5*IQR) |
        (listings[col] > Q3 + 1.5*IQR)
    ]

    print(col, len(outliers))

accommodates 35
bedrooms 30
number_of_reviews 45


In [ ]:
invalid_lat = listings[
    (listings["latitude"] < -90) |
    (listings["latitude"] > 90)
]

invalid_lon = listings[
    (listings["longitude"] < -180) |
    (listings["longitude"] > 180)
]

In [ ]:
calendar["price"] = (
    calendar["price"]
    .replace(
        r"[\$,]",
        "",
        regex=True
    )
    .astype(float)
)

In [ ]:
calendar["date"] = pd.to_datetime(
    calendar["date"]
)

reviews["date"] = pd.to_datetime(
    reviews["date"]
)

In [ ]:
listings["room_type"] = (
    listings["room_type"]
    .str.strip()
    .str.title()
)

In [ ]:
listings["review_scores_rating"] = (
    listings["review_scores_rating"]
    .fillna(
        listings[
            "review_scores_rating"
        ].median()
    )
)

In [ ]:
listings["valid_record"] = True

listings.loc[
    listings["latitude"].isna(),
    "valid_record"
] = False

In [ ]:
review_summary = (
    reviews
    .groupby("listing_id")
    .agg(
        review_count=("id","count")
    )
    .reset_index()
)

listing_master = listings.merge(
    review_summary,
    left_on="id",
    right_on="listing_id",
    how="left"
)

In [ ]:
calendar["occupied"] = (
    calendar["available"] == "f"
)

occupancy = (
    calendar.groupby("listing_id")
    ["occupied"]
    .mean()
    .reset_index()
)

occupancy.rename(
    columns={
        "occupied":
        "occupancy_rate"
    },
    inplace=True
)

In [ ]:
revenue = (
    calendar
    .assign(
        revenue=lambda x:
        x["price"] *
        x["occupied"]
    )
    .groupby("listing_id")
    ["revenue"]
    .sum()
    .reset_index()
)

In [ ]:
neighbourhood_stats = (
    listing_master
    .groupby(
        "neighbourhood_cleansed"
    )
    .agg({
        "review_scores_rating":
        "mean"
    })
)

In [ ]:
today = pd.Timestamp.today()

listing_master["host_tenure"] = (
    today -
    pd.to_datetime(
        listing_master["host_since"]
    )
).dt.days / 365

In [ ]:
!pip install duckdb

In [ ]:
import duckdb

con = duckdb.connect(
    "airbnb.duckdb"
)

In [ ]:
con.execute("""
CREATE TABLE dim_listing AS
SELECT
    id AS listing_id,
    name,
    room_type,
    property_type,
    accommodates
FROM listing_master;
""")

In [ ]:
con.execute("""
CREATE TABLE dim_neighbourhood AS
SELECT DISTINCT
    neighbourhood_cleansed AS neighbourhood
FROM listing_master;
""")

In [ ]:
print(listing_master.columns.tolist())

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minim

In [ ]:
listings.columns[listings.columns.str.contains("revenue|occup|review", case=False)]

Index(['number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
       'number_of_reviews_ly', 'estimated_occupancy_l365d',
       'estimated_revenue_l365d', 'first_review', 'last_review',
       'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value', 'reviews_per_month'],
      dtype='object')

In [ ]:
con.execute("""
CREATE TABLE fact_listing_metrics AS
SELECT
    id AS listing_id,
    neighbourhood_cleansed,
    estimated_occupancy_l365d,
    estimated_revenue_l365d,
    number_of_reviews,
    review_scores_rating
FROM listing_master;
""")

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_neighbourhood AS
SELECT DISTINCT
    neighbourhood_cleansed
FROM listing_master;
""")

In [ ]:
CONFIG = {
    "city":"houston",
    "input_path":"data/",
    "output_path":"output/"
}

In [ ]:
import logging

logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO
)

In [ ]:
try:
    listings = pd.read_csv(
        "/content/drive/MyDrive/Data Engineering/listings.csv"
    )

except Exception as e:

    logging.error(e)

In [ ]:
metadata = pd.DataFrame({
    "dataset":[
        "listings",
        "reviews"
    ],
    "ingestion_time":[
        pd.Timestamp.now(),
        pd.Timestamp.now()
    ]
})